# 04. Model XGBoost
XGBoost models implementation.

In [ ]:
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import StandardScaler
import notebook_const

from src import utils
from src import const

In [ ]:
# Create sequences or reuse cached tensors
data = utils.load_saved_tensors()

metadata = data['metadata']
n_past_trips = metadata['n_past_trips']
lag_columns = metadata['lag_columns']
train_data, test_data = data['train'], data['test']

X_delays_train, X_features_train, X_agg_train, y_train = \
    train_data['X_delays_train'], train_data['X_features_train'], train_data['X_agg_train'], train_data['y_train']
X_delays_test, X_features_test, X_agg_test, y_test = \
    test_data['X_delays_test'], test_data['X_features_test'], test_data['X_agg_test'], test_data['y_test']
n_stops = data['n_stops']

In [ ]:
# Prepare Data for XGBoost (aligned with LSTM preprocessing)
# Scale delays separately (like LSTM's delay_scaler)
delay_scaler = StandardScaler()
X_delays_train_flat = X_delays_train.reshape(len(X_delays_train), -1)
X_delays_test_flat = X_delays_test.reshape(len(X_delays_test), -1)
X_delays_train_scaled = delay_scaler.fit_transform(X_delays_train_flat)
X_delays_test_scaled = delay_scaler.transform(X_delays_test_flat)

# Scale y with the same delay scaler
y_train_scaled = delay_scaler.transform(y_train)
y_test_scaled = delay_scaler.transform(y_test)

# Scale features separately (like LSTM's feature_scaler)
feature_scaler = StandardScaler()
X_combined_train = np.concatenate([X_features_train, X_agg_train], axis=1)
X_combined_test = np.concatenate([X_features_test, X_agg_test], axis=1)
X_combined_train_scaled = feature_scaler.fit_transform(X_combined_train)
X_combined_test_scaled = feature_scaler.transform(X_combined_test)

# Concatenate scaled delays and scaled features
X_train_scaled = np.concatenate([X_delays_train_scaled, X_combined_train_scaled], axis=1)
X_test_scaled = np.concatenate([X_delays_test_scaled, X_combined_test_scaled], axis=1)

stop sequenceごと版

In [ ]:
# Train XGBoost (Per Stop)
evaluation_results = []

print("Training XGBoost...")
y_pred_xgb_all = []

xgb_params = {
    "n_estimators": 100,
    "max_depth": 5,
    "learning_rate": 0.1,
    "n_jobs": -1,
    "random_state": 42
}

for stop_idx in range(n_stops):
    model = xgb.XGBRegressor(**xgb_params)
    model.fit(X_train_scaled, y_train_scaled[:, stop_idx])
    pred = model.predict(X_test_scaled)
    y_pred_xgb_all.append(pred)

y_pred_xgb_scaled = np.array(y_pred_xgb_all).T
y_pred_xgb_all = delay_scaler.inverse_transform(y_pred_xgb_scaled)

result_xgb = utils.evaluate_model(
    y_test, y_pred_xgb_all,
    model_name="XGBoost (Per Stop)",
    config={"params": xgb_params, "n_past_trips": n_past_trips, "scaler": "StandardScaler"}
)
evaluation_results.append(result_xgb)
print(result_xgb.summary())

全stop版

In [ ]:
# Train XGBoost (All Stops)
print("Training XGBoost (All Stops)...")

model = xgb.XGBRegressor(**xgb_params)
model.fit(X_train_scaled, y_train_scaled)
y_pred_scaled = model.predict(X_test_scaled)
y_pred = delay_scaler.inverse_transform(y_pred_scaled)

result_xgb = utils.evaluate_model(
    y_test, y_pred,
    model_name="XGBoost (All Stops)",
    config={"params": xgb_params, "n_past_trips": n_past_trips, "scaler": "StandardScaler"}
)
evaluation_results.append(result_xgb)
print(result_xgb.summary())

In [ ]:
# Model Comparison Table and Save Results
utils.display_and_save_results(evaluation_results, const.EVALUATION_RESULTS_XGBOOST)